In [218]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
import statsmodels.api as sm
import statsmodels.formula.api as smf

np.random.seed(1)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import LeaveOneOut, KFold, train_test_split
from sklearn.metrics import accuracy_score
import random
from timeit import default_timer as timer

In [219]:
df = pd.read_csv('https://raw.githubusercontent.com/sukhjitsehra/datasets/master/CP322/Default.csv')
df.head(10)

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879
5,No,Yes,919.588530,7491.558572
6,No,No,825.513331,24905.226578
7,No,Yes,808.667504,17600.451344
8,No,No,1161.057854,37468.529288
9,No,No,0.000000,29275.268293


In [220]:
# Note: factorize() returns two objects: a label array and an array with the unique values.
# We are only interested in the first object. 
df['default2'] = df.default.factorize()[0]
df['student2'] = df.student.factorize()[0]
df.head(3)

,default,student,balance,income,default2,student2
0,No,No,729.526495,44361.625074,0,0
1,No,Yes,817.180407,12106.134700,0,1
2,No,No,1073.549164,31767.138947,0,0


In [222]:
X = df[['balance', 'income', 'student2']]
y=df.default2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=42)

In [ ]:
class ClassifierEvaluator:
    def __init__(self, clf, X, y):
        self.clf = clf
        self.X = X
        self.y = y
        
    def evaluate_loocv(self):
        """Leave One Out Cross-Validation"""
        start= timer()
        loocv = LeaveOneOut()
        scores = []
        for train_index, test_index in loocv.split(self.X):
            X_train, X_test = self.X.iloc[train_index], self.X.iloc[test_index]
            y_train, y_test = self.y.iloc[train_index], self.y.iloc[test_index]
            self.clf.fit(X_train, y_train)
            y_pred = self.clf.predict(X_test)
            score = accuracy_score(y_test, y_pred)
            scores.append(score)
        end = timer()
        return np.mean(scores), end-start
    
    def evaluate_kfold(self, k=5):
        """K-fold Cross-Validation"""
        kf = KFold(n_splits=k)
        scores = []
        for train_index, test_index in kf.split(self.X):
            X_train, X_test = self.X.iloc[train_index], self.X.iloc[test_index]
            y_train, y_test = self.y.iloc[train_index], self.y.iloc[test_index]
            self.clf.fit(X_train, y_train)
            y_pred = self.clf.predict(X_test)
            score = accuracy_score(y_test, y_pred)
            scores.append(score)
        return np.mean(scores)
    
    def evaluate_validation_set(self, test_size=0.2):
        """Validation Set Approach"""
        X_train, X_val, y_train, y_val = train_test_split(self.X, self.y, test_size=test_size)
        self.clf.fit(X_train, y_train)
        y_pred = self.clf.predict(X_val)
        return accuracy_score(y_val, y_pred)
    
    def evaluate_bootstrap(self, n_samples=1000):
        """Bootstrap Resampling"""
        scores = []
        for _ in range(n_samples):
            # Draw a random sample from the data with replacement
            sample_index = np.random.choice(self.X.index, len(self.X), replace=True)
            X_sample, y_sample = self.X.loc[sample_index], self.y.loc[sample_index]
            self.clf.fit(X_sample, y_sample)
            y_pred = self.clf.predict(X_sample)
            score = accuracy_score(y_sample, y_pred)
            scores.append(score)
        return np.mean(scores)
if __name__ == '__main__':
    clf = LogisticRegression()
    evaluator = ClassifierEvaluator(clf, X, y)
    # call the evaluate_loocv() method to get the LOOCV score
    loocv_score, time = evaluator.evaluate_loocv()
    print("LOOCV score: ", loocv_score)
    print("Time taken by the function to compute the LOOCV is", time)

    # call the evaluate_kfold() method to get the K-fold CV score
    kfold_score = evaluator.evaluate_kfold()
    print("K-fold CV score: ", kfold_score)

    # call the evaluate_validation_set() method to get the validation set score
    validation_set_score = evaluator.evaluate_validation_set()
    print("Validation set score: ", validation_set_score)

    # call the evaluate_bootstrap() method to get the bootstrap resampling score
    bootstrap_score = evaluator.evaluate_bootstrap()
    print("Bootstrap score: ", bootstrap_score)

/Volumes/data/SystemSoftwares/anaconda3/envs/r_40/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LOOCV score:  0.9675
Time taken by the function to compute the LOOCV is 1029.0343556519947
K-fold CV score:  0.9692000000000001
Validation set score:  0.967
Bootstrap score:  0.9675573999999999


In [223]:
import unittest
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

class TestClassifierEvaluator(unittest.TestCase):
    def setUp(self):
        self.X, self.y = make_classification(n_classes=2, n_samples=1000)
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(self.X, self.y, test_size=0.2)
        self.clf = LogisticRegression()
        self.evaluator = ClassifierEvaluator(self.clf, self.X_train, self.y_train)

    # def test_loocv(self):
    #     loocv_score = self.evaluator.evaluate_loocv()
    #     self.assertGreaterEqual(loocv_score, 0.7)
        
    def test_kfold(self):
        kfold_score = self.evaluator.evaluate_kfold()
        self.assertGreaterEqual(kfold_score, 0.7)

    def test_validation_set(self):
        validation_set_score = self.evaluator.evaluate_validation_set()
        self.assertGreaterEqual(validation_set_score, 0.7)
        
    # def test_bootstrap(self):
    #     bootstrap_score = self.evaluator.evaluate_bootstrap()
    #     self.assertGreaterEqual(bootstrap_score, 0.7)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

EE
ERROR: test_kfold (__main__.TestClassifierEvaluator)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ssehra\AppData\Local\Temp/ipykernel_23500/979005358.py", line 11, in setUp
    self.evaluator = ClassifierEvaluator(self.clf, self.X_train, self.y_train)
NameError: name 'ClassifierEvaluator' is not defined

ERROR: test_validation_set (__main__.TestClassifierEvaluator)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\ssehra\AppData\Local\Temp/ipykernel_23500/979005358.py", line 11, in setUp
    self.evaluator = ClassifierEvaluator(self.clf, self.X_train, self.y_train)
NameError: name 'ClassifierEvaluator' is not defined

----------------------------------------------------------------------
Ran 2 tests in 0.017s

FAILED (errors=2)


In [224]:
df = pd.read_csv('https://raw.githubusercontent.com/sukhjitsehra/datasets/master/CP322/Auto.csv', na_values='?').dropna() # drop columns with NaN
print("Shape of dataframe: " + str(df.shape))
df.head()

Shape of dataframe: (392, 9)


,mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,1,ford torino


#### The Validation Set Approach

The LinearRegression module (and other modules) in sklearn requires both the `fit()` and `predict()` to be an array with shape `(n_samples, n_features)`. Since we are only using one feature in this example from the book, we use `reshape` to get our data in the correct shape. If we use >1 predictors we do not need to take the `reshape` path.

In [380]:
X = df['horsepower'].values.reshape(-1,1)
y = df['mpg'].values.reshape(-1,1)

# split data. adjust 'random_state' for new seed
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=1)

# instantiate and train model
clf = LinearRegression()
clf.fit(X_train, y_train)

LinearRegression()

In [381]:
# predict on our test set
y_pred = clf.predict(X_test)

# get MSE on our test set
mean_squared_error(y_test, y_pred)

24.80212062059356

#### 5.3.2 Leave-One-Out Cross-Validation

In [226]:
from sklearn.model_selection import KFold
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_score

clf = LinearRegression()
loo = LeaveOneOut()
loo.get_n_splits(X) # = n

10000

In [228]:
scores = cross_val_score(clf, X, y, cv=loo, scoring='neg_mean_squared_error')
print(abs(scores.mean()))

0.028236032064781244


#### 5.3.3 k-fold Cross-Validation

In [ ]:
# k=10
kf = KFold(n_splits=10, random_state=1, shuffle=True)
scores = cross_val_score(clf, X, y, cv=kf, scoring='neg_mean_squared_error')
print(abs(scores.mean()))
print(abs(scores))

NameError: name 'clf' is not defined

In [229]:
# polynomials loop for horsepower

from sklearn.preprocessing import PolynomialFeatures
import time

p_order = np.arange(1,11)
r_state = np.arange(0,10)

# LeaveOneOut CV
regr = LinearRegression()
loo = LeaveOneOut()
loo.get_n_splits(df)
scores = list()

# loop through polynomials to the the 10th degree
start = time.time()
for i in range(1,11):
    poly = PolynomialFeatures(i)
    X_poly = poly.fit_transform(df.horsepower.values.reshape(-1,1))
    score = cross_val_score(regr, X_poly, df.mpg, cv=loo, scoring='neg_mean_squared_error').mean()
    scores.append(abs(score))
end = time.time()    
print(scores)
print("Took: " + str(end-start) + " seconds to run.")
#loo takes much longer to run than k=10

[24.231513517929226, 19.24821312448972, 19.334984064079364, 19.424430309616614, 19.033204805230227, 19.003944342792217, 19.12560636282337, 19.224181705126682, 19.13396345862963, 18.94648397930701]
Took: 6.247650623321533 seconds to run.


#### 5.3.4 The Bootstrap

In [234]:
dfp= pd.read_csv("https://raw.githubusercontent.com/sukhjitsehra/datasets/master/CP322/Portfolio.csv", index_col=0).dropna()

In [238]:
print(dfp.shape)
dfp.head()

(100, 1)


,Y
X,
-0.895251,-0.234924
-1.562454,-0.885176
-0.417090,0.271888
1.044356,-0.734198
-0.315568,0.841983


In [237]:
def alpha_fn(data, r, c):
    X = data.X[r:c]
    Y = data.Y[r:c]
    return ((np.var(Y)-np.cov(X,Y)[0,1])/(np.var(X)+np.var(Y) - 2 * np.cov(X,Y)[0,1]))

alpha_fn(dfp, 0, 100)

AttributeError: 'DataFrame' object has no attribute 'X'

In [239]:
# cleaner version
def alpha_fn(data, index):
    X = data.X[index]
    Y = data.Y[index]
    return (np.var(Y) - np.cov(X,Y)[0,1])/(np.var(X) + np.var(Y) - 2 * np.cov(X, Y)[0,1])

alpha_fn(dfp, list(range(100)))

AttributeError: 'DataFrame' object has no attribute 'X'

In [ ]:
# numpy version of random sampling for bootstrapping

np.random.seed(1)
alpha_fn(dfp, np.random.choice(list(range(100)), size=100))

In [240]:
# sklearn version of sampling for bootstrapping

from sklearn.utils import resample

def alpha_fn(data, r, c):
    data = resample(data, n_samples=100, random_state=1)
    X = data.X[r:c]
    Y = data.Y[r:c]
    return ((np.var(Y)-np.cov(X,Y)[0,1])/(np.var(X)+np.var(Y) - 2 * np.cov(X,Y)[0,1]))

alpha_fn(dfp, 0, 100)

AttributeError: 'DataFrame' object has no attribute 'X'